# `edge_eeg` Colab CPU High-RAM Cache Builder
专门用于在 Colab CPU 实例上拉取单被试 (`sub-001`) LibriBrain 数据、在本地高速 NVMe (`/content/local_cache`) 生成特征缓存，并打包为单文件 `.tar` 存入 Google Drive。

In [ ]:
# === Cell 1: 挂载 Drive、拉取 Git 代码、建立本地软链桥 (/content/studies_links) ===
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive")
DRIVE_WS = Path("/content/drive/MyDrive/EdgeEEG_Workspace")
for sub in ("data", "archives", "runs"):
    (DRIVE_WS / sub).mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/<YOUR_USERNAME>/EmbeddedBrainDecoding.git"
CODE_DIR = Path("/content/code")
if not CODE_DIR.exists():
    !git clone {REPO_URL} {CODE_DIR}
else:
    !git -C {CODE_DIR} pull --ff-only

LINKS_DIR = Path("/content/studies_links")
LINKS_DIR.mkdir(parents=True, exist_ok=True)
for ds_name in ("libribrain", "auditory_eeg"):
    target = DRIVE_WS / "data" / ds_name
    target.mkdir(parents=True, exist_ok=True)
    link_path = LINKS_DIR / ds_name
    if link_path.is_symlink() or link_path.exists():
        link_path.unlink()
    os.symlink(target, link_path)
print("Symlink bridge ready:", list(LINKS_DIR.iterdir()))

In [ ]:
# === Cell 2: 初始化兼容 Python 3.13+ 的安全虚拟环境 ===
!python3 /content/code/scripts/colab_setup.py --venv /content/eeg-venv --repo /content/code
PYTHON = "/content/eeg-venv/bin/python"

In [ ]:
# === Cell 3: 在本地 NVMe (/content/local_cache) 上预计算单被试特征缓存 ===
import os
os.environ["EDGEEEG_STUDIES"] = "/content/studies_links"
os.environ["EDGEEEG_CACHE"] = "/content/local_cache"
os.environ["EDGEEEG_RESULTS"] = "/content/drive/MyDrive/EdgeEEG_Workspace/runs"

MODALITY = "meg"
SUBJECT_ARGS = "--subject 001"

!{PYTHON} -u -m edge_eeg cache --modality {MODALITY} --stage vad {SUBJECT_ARGS} --cpu-threads 4
!{PYTHON} -u -m edge_eeg cache --modality {MODALITY} --stage phoneme {SUBJECT_ARGS} --cpu-threads 4

In [ ]:
# === Cell 4: 将本地 NVMe 缓存打包归档至 Google Drive ===
ARCHIVE_PATH = f"/content/drive/MyDrive/EdgeEEG_Workspace/archives/features_{MODALITY}_sub001.tar"
!{PYTHON} -u /content/code/scripts/cache_archive.py pack \
    --cache-dir /content/local_cache \
    --archive {ARCHIVE_PATH}